# 08 — End-to-end H&E/CD8 workflow

Alignment → matched patches → stain normalization → cell counting. Each step
takes the previous step's result directly, and every output folder records
its settings in `rocqipath.json`. Enable one stage at a time.

```bash
python -m pip install -e ".[extraction,orb,stain,cellcount,viz]"
```

In [ ]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository whether Jupyter starts at its root or in how_to_use/."""
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "rocqipath").is_dir():
            return candidate
    return here


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"          # your slides (kept out of git)
RESULTS_ROOT = PROJECT_ROOT / "results"    # outputs (kept out of git)
DEMO_ROOT = PROJECT_ROOT / "notebook_demo_outputs"  # synthetic examples

import rocqipath as rp

print(f"RocqiPath {rp.__version__}")
print(f"Data    : {DATA_ROOT}")
print(f"Results : {RESULTS_ROOT}")

In [ ]:
PAIRS_ROOT = DATA_ROOT / "pairs"
OUT = RESULTS_ROOT / "he_cd8"

ALIGN = dict(pair_folders=["CD8"], reference_name="he", moving_name="cd8", backend="orb",
             target_magnification=20.0, qc_enabled=True)
PATCHES = dict(patch_size=512, tissue_threshold=0.50, moving_name="cd8", max_workers=4)
STAIN = dict(method="macenko", stains=["he"], max_train_patches=500)
COUNT = dict(label="CD8", target_magnification=20.0, min_cell_area=50, max_cell_area=1000)

RUN_DRY_RUN = False
RUN_ALIGNMENT = False
RUN_PATCHES = False
RUN_STAIN = False
RUN_COUNTS = False

## Stage 1 — Pairing and alignment

In [ ]:
if RUN_DRY_RUN:
    rp.align(PAIRS_ROOT, OUT / "aligned", **ALIGN, dry_run=True)

aligned = rp.align(PAIRS_ROOT, OUT / "aligned", **ALIGN) if RUN_ALIGNMENT else OUT / "aligned"
print(aligned if not RUN_ALIGNMENT else f"{len(aligned.by_role('aligned'))} aligned slides")

## Stage 2 — Matched patches

`aligned` is either the result above or the output folder from an earlier
session; both work because the folder records what it holds.

In [ ]:
if RUN_PATCHES:
    patches = rp.extract_patches(aligned, OUT / "patches", **PATCHES)
    print(patches.summary["processed"], "cases,", len(patches), "patch images")
else:
    patches = OUT / "patches"
    print("Patch extraction disabled.")

## Stage 3 — H&E stain normalization (optional)

Fit on training patches only in a real study, then reuse the weights.

In [ ]:
if RUN_STAIN:
    weights = rp.train_stain_normalizer(patches, OUT / "stain", **STAIN)
    normalized = rp.normalize_stain(patches, OUT / "stain", normalizer=weights, stains=STAIN["stains"])
    print(normalized.summary)
else:
    print("Stain normalization disabled.")

## Stage 4 — CD8 cell counting

The aligned slides carry a manifest with their magnification, so no source
magnification is needed.

In [ ]:
if RUN_COUNTS:
    counts = rp.count_cells(aligned, OUT / "counts", **COUNT)
    for slide in counts.summary["results"]:
        print(f"{slide['slide']:45s} {slide['total_positive']:>8,} CD8+ cells")
else:
    print("Cell counting disabled.")

## Stage 5 — Provenance

Every stage already wrote its settings, inputs and outputs to
`<stage>/rocqipath.json`. Read them back at any time:

In [ ]:
from rocqipath.io.manifest import read_run_manifest

for stage in ("aligned", "patches", "stain", "counts"):
    folder = OUT / stage
    if (folder / "rocqipath.json").exists():
        for result in read_run_manifest(folder):
            print(f"{stage:8s} {result.workflow:24s} {len(result):>6} files")

## Scientific QC gates

1. **Alignment** — review several regions and tissue boundaries.
2. **Patches** — confirm paired morphology; check the edge/background rate.
3. **Normalization** — verify nuclei and DAB signal are preserved.
4. **Counting** — validate masks against manual review.
5. **Provenance** — keep the `rocqipath.json` files, package version and cohort
   exclusions with the analysis.